# Research models for turbofan RUL

The temporal CNN uses dilated convolutions and the GRU uses gated recurrent memory over the same 30 observed cycles. Both add the latest sensor reading to the regression head. The SNN uses 64 LIF neurons and receives either ground truth alone, CNN targets, GRU targets, or an equal average of the two frozen teachers. Each guided student uses the same 60% ground-truth and 40% teacher objective. The supervised student controls for capacity and the training target.

Training uses raw RUL divided by 125 for numerical scale. The original experiments capped training labels, so this comparison changes the target as well as the architecture. Checkpoint selection uses raw validation RMSE on held-out training engines. Test files are opened after all selections have been saved. This first experiment uses seed 42 across four subsets.

[Convolutional RUL prediction](https://escholarship.org/uc/item/5ns8r3fs) · [Gated recurrent models](https://aclanthology.org/D14-1179/) · [Knowledge distillation](https://arxiv.org/abs/1503.02531)

In [1]:
from pathlib import Path
import subprocess
import sys
from zipfile import ZipFile
import torch
from google.colab import drive

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive")
data_root = drive_root / "ML projec/SpikeDistill-RUL/artifacts/data"
bundle = drive_root / "SpikeDistill-RUL - Project Demo/04 - Research extensions/Research_Models_Source.zip"
repo = Path("/content/spikedistill_research_models")
with ZipFile(bundle) as source:
    source.extractall(repo)
run_root = drive_root / "ML projec/SpikeDistill-RUL/colab_runs/research_teachers_v1"
device = "cuda" if torch.cuda.is_available() else "cpu"
subsets = ["FD001", "FD002", "FD003", "FD004"]
seeds = ["42"]
print(f"Device {device}. Checkpoints and scores are saved to Drive.")

Mounted at /content/drive
Device cuda. Checkpoints and scores are saved to Drive.


## Train and select on validation

In [2]:
command = [sys.executable, "-u", "-m", "research_extensions.teacher_student",
           "--data-root", str(data_root), "--output", str(run_root),
           "--subsets", *subsets, "--seeds", *seeds, "--device", device]
subprocess.run([*command, "--phase", "train"], cwd=repo, check=True)

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'research_extensions.teacher_student', '--data-root', '/content/drive/MyDrive/ML projec/SpikeDistill-RUL/artifacts/data', '--output', '/content/drive/MyDrive/ML projec/SpikeDistill-RUL/colab_runs/research_teachers_v1', '--subsets', 'FD001', 'FD002', 'FD003', 'FD004', '--seeds', '42', '--device', 'cuda', '--phase', 'train'], returncode=0)

## Evaluate the frozen checkpoints on the official test set

In [3]:
subprocess.run([*command, "--phase", "test"], cwd=repo, check=True)

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'research_extensions.teacher_student', '--data-root', '/content/drive/MyDrive/ML projec/SpikeDistill-RUL/artifacts/data', '--output', '/content/drive/MyDrive/ML projec/SpikeDistill-RUL/colab_runs/research_teachers_v1', '--subsets', 'FD001', 'FD002', 'FD003', 'FD004', '--seeds', '42', '--device', 'cuda', '--phase', 'test'], returncode=0)

## Paired teacher and student tables

The two-teacher student is compared with the equal-average teacher prediction. A negative student-minus-teacher difference means the student achieved lower RMSE. The comparison uses the same engine endpoints and raw cycle targets for every model.

In [4]:
sys.path.insert(0, str(repo))
from research_extensions.teacher_student_report import report

report(run_root, subsets, [int(seed) for seed in seeds], repo / "results/test_metrics_per_seed.csv")

Official raw test RMSE for each teacher–student pair


,subset,seed,split,target,teacher,student,teacher_RMSE,student_RMSE,student_minus_teacher,student_minus_supervised
6,FD001,42,test,raw,temporal_cnn,snn_from_cnn,19.746,22.156,2.411,-2.750
7,FD001,42,test,raw,gru_memory,snn_from_gru,22.455,23.095,0.640,-1.812
8,FD001,42,test,raw,teacher_average,snn_from_both,20.182,22.891,2.709,-2.015
18,FD002,42,test,raw,temporal_cnn,snn_from_cnn,26.265,27.409,1.145,-1.359
19,FD002,42,test,raw,gru_memory,snn_from_gru,25.634,27.506,1.872,-1.262
20,FD002,42,test,raw,teacher_average,snn_from_both,25.359,28.101,2.742,-0.668
30,FD003,42,test,raw,temporal_cnn,snn_from_cnn,30.060,34.535,4.475,-8.398
31,FD003,42,test,raw,gru_memory,snn_from_gru,26.662,34.465,7.804,-8.468
32,FD003,42,test,raw,teacher_average,snn_from_both,27.333,36.691,9.357,-6.243
42,FD004,42,test,raw,temporal_cnn,snn_from_cnn,28.381,32.265,3.884,-1.904


Official raw test metrics for every new model


,subset,seed,model,RMSE,MAE,MSE,RAE_pct,R2,cmapss_score,bias_cycles
14,FD001,42,temporal_cnn,19.746,14.233,389.886,38.712,0.774,1534.357,1.818
15,FD001,42,gru_memory,22.455,16.174,504.209,43.990,0.708,8184.304,8.594
16,FD001,42,snn_supervised,24.906,19.165,620.328,52.126,0.641,4907.190,11.662
17,FD001,42,snn_from_cnn,22.156,16.505,490.909,44.891,0.716,3493.432,6.930
18,FD001,42,snn_from_gru,23.095,17.112,533.369,46.541,0.691,3407.740,7.358
19,FD001,42,snn_from_both,22.891,17.743,523.997,48.257,0.697,2768.751,9.228
20,FD001,42,teacher_average,20.182,14.206,407.304,38.639,0.764,3096.682,5.206
42,FD002,42,temporal_cnn,26.265,18.006,689.830,39.761,0.761,9453.675,-1.254
43,FD002,42,gru_memory,25.634,17.722,657.117,39.134,0.773,9064.910,-4.287
44,FD002,42,snn_supervised,28.768,20.281,827.610,44.786,0.714,15512.302,-2.560


Original seed-matched raw test reference


,model,ann_to_snn,attention_bilstm,bilstm,cnn_lstm,deepnn,gru,learnable_snn,lstm,mean,rnn,small_lstm_kd,snn,spike_output_kd,spike_temporal_kd,spikedistill,widenn
subset,seed,,,,,,,,,,,,,,,,
FD001,42,47.208,15.729,16.311,16.907,17.026,15.846,18.624,16.134,41.981,15.633,15.473,18.908,17.751,18.194,17.548,16.369
FD002,42,51.666,29.698,28.455,29.555,28.716,28.543,28.840,29.827,53.780,29.826,30.020,31.026,28.786,29.638,28.962,28.202
FD003,42,31.748,14.166,15.301,17.303,14.552,16.466,21.889,17.879,43.763,17.955,17.139,21.663,21.092,19.631,20.402,16.243
FD004,42,60.966,27.695,28.335,28.433,28.045,26.567,30.288,26.294,54.560,27.647,27.100,30.801,29.485,28.088,28.492,29.880
